# Jev-CeNN — Selective W512 Triple Compression, Streaming Distillation and Sealed Holdout

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_W512_Selective_3to1_Heldout_Colab.ipynb)

## Objective

The previous full-encoder W512 28→10 experiment **FAILED**: final agreement was **65.625%**, gold accuracy **57.5%** versus teacher **88.75%**, and final batch-2 inference was **3.6% slower**. Nevertheless, CeNN helped many groups and the isolated 24–26 group reached 0.991 local cosine. Local quality at original teacher inputs did not predict quality on real compressed-prefix inputs.

This notebook **does NOT force nine replacements**. It independently trains each proposed 3→1 W512 CeNN superlayer but **accepts it only** when the actual cumulative full-model teacher agreement, gold accuracy and *student-prefix* fidelity pass explicit DEV-selection gates. All other original layers, three full FFNs, and native global/sliding attention are preserved wherever a candidate fails. It can legitimately accept **zero** groups.

### Architecture

Original 28 complete ModernBERT encoder blocks. Each accepted (global + sliding + sliding) triple becomes ONE two-stage attention/active CeNN superlayer with W512 local SDPA, W45 stride-23 dilated attention, rank-96 correction and one FFN. Slots +1,+2 become bypasses. Unaccepted triples remain original; layer 27 stays original. It is a selective rather than forced 28→10 experiment.

### Controls addressing the previous run

- **Fresh dynamic training data**: random, non-repeating cycles through **hundreds of TRAIN decisions**, captured from the frozen original teacher, not a cache of 64 items replayed 900 times.
- **Real student-prefix inputs every training step**: capture activations entering the candidate from the **current, partially compressed** encoder; teacher group targets are computed for the exact same input cases.
- **Per-group end-to-end recovery**: install the candidate first, optimize it through the *entire frozen remaining encoder and original task head*, using teacher KL/ranking, supervised labels and final-encoder hidden-state alignment.
- **DEV-only acceptance**: require compressed-prefix cosine ≥0.99, NMSE ≤0.025, cumulative teacher agreement ≥0.99, <=1.5-point gold drop versus teacher, and bounded incremental degradation. Roll back the three original modules on any failure. Track acceptance/rejection reasons and latency for every attempted group.
- **Case-disjoint partitions**: TRAIN for optimizer updates, DEV-PROBE for selecting local checkpoints, DEV-GATE for task feedback and candidate acceptance, and a strictly separate final HOLDOUT for **one-time report only**. All are derived from the benchmark's TRAIN split; the official benchmark TEST remains untouched. These holdout cases are independent of this experiment's optimization and checkpoint selection; they are **not proof that the source model never saw similar data in its own pretraining**.
- **Paired CUDA benchmarking**: timed native teacher and selected student using identical tensors, both batch 1 and 2 and short/long sequence examples, with warmup and CUDA events.
- **Scientific rollback**: always save diagnostics, even if every candidate is rejected. Final model is not published, and **no claim of 0.99 accuracy or GPU speedup** is made before actual GPU results.

## Run order

Install → load teacher and partition case IDs → initialize W512 replacement → stream TRAIN teacher+rollout pairs → local PROBE selection → install and fine-tune on changed full model → measure DEV-GATE fidelity/decisions/speed → **commit or rollback** → next group → sealed HOLDOUT once → save checkpoints, CSV/JSON, and figures.

**Optional:** mount Google Drive and set \`OUTPUT_DIR\` to a Drive folder before starting expensive training. A standard Colab GPU is required.


In [ ]:
#@title 1. Install pinned teacher and latest selective CeNN dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. W512 selective thresholds, teacher and CUDA setup
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
MAX_LEN=512
LOCAL_WINDOW=512 #@param {type:"integer"}
LOCAL_RADIUS=22
DILATION_STRIDE=23
DILATED_RADIUS=22
CORRECTION_RANK=96 #@param {type:"integer"}
TRAIN_CASES=700 #@param {type:"integer"}
DEV_CASES=220 #@param {type:"integer"}
HOLDOUT_CASES=200 #@param {type:"integer"}
PROBE_CASES=80 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
PROBE_BATCHES=12 #@param {type:"integer"}
DEV_GATE_DECISIONS=240 #@param {type:"integer"}
HOLDOUT_DECISIONS=500 #@param {type:"integer"}
CENN_STEPS_PER_GROUP=60 #@param {type:"integer"}
STEPS_PER_GROUP=450 #@param {type:"integer"}
FEEDBACK_STEPS_PER_GROUP=100 #@param {type:"integer"}
PROBE_EVERY=50 #@param {type:"integer"}
FEEDBACK_EVERY=20 #@param {type:"integer"}
COSINE_WEIGHT=3.0
INTERMEDIATE_WEIGHT=0.10
CENN_AUX_WEIGHT=0.02
GROUP_LR=0.0002
QKV_LR=0.00002
FEEDBACK_LR=0.000008 #@param {type:"number"}
HIDDEN_ALIGN_WEIGHT=0.10 #@param {type:"number"}
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
REQUIRE_CANDIDATE_SPEEDUP=False #@param {type:"boolean"}
USE_FP16_AMP=False #@param {type:"boolean"}
BENCH_REPEATS=10 #@param {type:"integer"}
OUTPUT_DIR=Path("/content/Jev_CeNN_W512_Selective")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert MAX_LEN==512 and LOCAL_WINDOW==512 and BATCH>0
assert TRAIN_CASES>0 and DEV_CASES>PROBE_CASES>0 and HOLDOUT_CASES>0
assert 0<HOLDOUT_DECISIONS and 0<DEV_GATE_DECISIONS
assert PROBE_BATCHES>=2 and STEPS_PER_GROUP>=1
assert 0<=CENN_STEPS_PER_GROUP and 0<=FEEDBACK_STEPS_PER_GROUP
assert PROBE_EVERY>=1 and FEEDBACK_EVERY>=1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert len(teacher.encoder.layers)==28 and GLOBAL_LAYERS==list(range(0,28,3))
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Two attention windows: exact W",LOCAL_WINDOW,"and W",2*DILATED_RADIUS+1," stride",DILATION_STRIDE,"| MAX_LEN:",MAX_LEN)
assert all(tuple(teacher.encoder.layers[i].attn.local_attention)==(64,64) for i in SLIDING_LAYERS)
print("Selective groups; all rejected groups stay native. Final HOLDOUT is sealed.")
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Strict case-disjoint TRAIN / PROBE / GATE / final HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred = repl(entry["x"], **entry["call_kwargs"])[0]
            require_finite(pred, "all-attention local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. Candidate triples and exact W512 attention mask
from tinycenn_lm.jev_cenn_full_encoder import (
    group_plan,preserved_tail,JevCeNNGroupSuperLayer,
    install_group,restore_group,installed_summary
)
from tinycenn_lm.jev_cenn_pairwindow_superlayer import (
    pair_reachable,pair_coverage_fraction,BypassModernBertLayer
)
GROUPS=group_plan(len(teacher.encoder.layers))
TAIL=preserved_tail(len(teacher.encoder.layers))
assert GROUPS==tuple((i,i+1,i+2) for i in range(0,27,3))
assert TAIL==(27,)
assert all(g[0] in GLOBAL_LAYERS and
           g[1] in SLIDING_LAYERS and g[2] in SLIDING_LAYERS for g in GROUPS)
assert pair_reachable(MAX_LEN,LOCAL_RADIUS,DILATION_STRIDE,DILATED_RADIUS)
print("SELECTIVE candidate order:",GROUPS,"| permanently native tail:",TAIL)
print("Full receptive-field coverage:",
      pair_coverage_fraction(MAX_LEN,LOCAL_RADIUS,DILATION_STRIDE,DILATED_RADIUS))
print("First attention exact W",LOCAL_WINDOW,
      "second dilated W",2*DILATED_RADIUS+1,
      "; dense-first theoretical pairs/head:",
      MAX_LEN*(LOCAL_WINDOW+2*DILATED_RADIUS+1))
assert all(type(student_eval.encoder.layers[i])==
           type(teacher.encoder.layers[i]) for i in range(28))


In [ ]:
#@title 5. Dynamic teacher triplets and ACTUAL partially compressed student-prefix activations
from tinycenn_lm.jev_cenn_full_encoder import stable_window_attention
import gc, csv, hashlib
def fresh_cycle(items, seed):
    """Shuffle all training decisions without replacement before reshuffling."""
    rng=random.Random(seed)
    order=list(range(len(items)))
    while True:
        rng.shuffle(order)
        for offset in range(0,len(order),BATCH):
            idx=order[offset:offset+BATCH]
            if len(idx)<BATCH:
                continue
            yield [items[j] for j in idx]
def _safe_copy(x):
    if x is None:return None
    y=x.detach().clone()
    if y.is_floating_point() and not torch.isfinite(y).all():
        raise FloatingPointError("Frozen-teacher/current-prefix activation is nonfinite")
    return y
def capture_triplet(start,chosen,include_rollout=True):
    """Same cases, original teacher targets, current student prefix inputs."""
    inp=direct_batch(chosen)
    captured={}
    def before(mod,args,kwargs):
        captured["x"]=_safe_copy(args[0])
        captured["position_ids"]=_safe_copy(kwargs.get("position_ids"))
    def first(mod,args,kwargs,out):
        captured["y_first"]=_safe_copy(out[0])
    def last(mod,args,kwargs,out):
        captured["y_end"]=_safe_copy(out[0])
    hooks=[
        teacher.encoder.layers[start].register_forward_pre_hook(before,with_kwargs=True),
        teacher.encoder.layers[start].register_forward_hook(first,with_kwargs=True),
        teacher.encoder.layers[start+2].register_forward_hook(last,with_kwargs=True),
    ]
    try:
        with torch.no_grad(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            teacher(**inp)
    finally:
        for h in hooks:h.remove()
    assert all(k in captured for k in ("x","y_first","y_end"))
    if include_rollout:
        got={}
        def hook(mod,args,kwargs):
            got["rollout"]=_safe_copy(args[0])
        h=student_eval.encoder.layers[start].register_forward_pre_hook(
            hook,with_kwargs=True)
        try:
            with torch.no_grad(),torch.autocast(
                    "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
                student_eval(**inp)
        finally:
            h.remove()
        if not got:
            raise RuntimeError("Failed to capture REAL compressed-prefix state")
        captured.update(got)
    else:
        captured["rollout"]=captured["x"]
    captured["valid"]=inp["attention_mask"].bool()
    return captured
def make_group_probe(start):
    entries=[]
    # Fixed DEV-PROBE: selection only; never the sealed HOLDOUT.
    for i in range(PROBE_BATCHES):
        chosen=probe_items[i*BATCH:(i+1)*BATCH]
        entries.append(capture_triplet(start,chosen))
    return entries
def check_tensor(value,name):
    if not torch.isfinite(value.float()).all():
        raise FloatingPointError(f"{name} has nonfinite values")
print("Dynamic TRAIN sampling will cover diverse cases; no 64-decision static cache.")


In [ ]:
#@title 6. Exact local fidelity and frozen teacher decision metrics (DEV-GATE only)
def cpu_snapshot(model):
    return {k:v.detach().to("cpu").clone() for k,v in model.state_dict().items()}
def pair_loss(pred,target,valid):
    a,b=pred.float(),target.float()
    va,vb=a[valid],b[valid]
    check_tensor(va,"student valid hidden")
    check_tensor(vb,"teacher valid hidden")
    difference=va-vb
    nmse=difference.square().sum()/vb.square().sum().clamp_min(1e-8)
    cosine=F.cosine_similarity(va,vb,dim=-1).mean()
    return nmse+COSINE_WEIGHT*(1-cosine),float(nmse.detach()),float(cosine.detach())
@torch.inference_mode()
def local_probe(model,entries,input_key="rollout"):
    model.eval()
    error=power=cs=ntokens=0.0
    for entry in entries:
        x=entry[input_key]
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            y=model(x,attention_mask=entry["valid"],
                    position_ids=entry["position_ids"])[0]
        good=entry["valid"]
        a,b=y.float()[good],entry["y_end"].float()[good]
        check_tensor(a,"local student")
        check_tensor(b,"local teacher")
        error+=float((a-b).square().sum())
        power+=float(b.square().sum())
        cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
        ntokens+=int(good.sum())
    return {"cosine":cs/max(ntokens,1),"nmse":error/max(power,1e-8),
            "tokens":int(ntokens)}
def local_rank(original,rollout):
    return (0.25*(original["nmse"]+COSINE_WEIGHT*(1-original["cosine"]))+
            0.75*(rollout["nmse"]+COSINE_WEIGHT*(1-rollout["cosine"])))
DEV_GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(DEV_GATE_ITEMS)==DEV_GATE_DECISIONS
def cache_teacher_decisions(items):
    records=[]
    for i in range(0,len(items),BATCH):
        chosen=items[i:i+BATCH]
        inp=direct_batch(chosen)
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=teacher(**inp)
        records.append({
            "batch":inp,
            "teacher_logits":logits.float().detach().clone(),
            "gold":torch.tensor([j["gold_index"] for j in chosen],
                                dtype=torch.long,device=device),
        })
    return records
DEV_GATE_RECORDS=cache_teacher_decisions(DEV_GATE_ITEMS)
@torch.inference_mode()
def quality_from_records(model,records):
    model.eval()
    n=agree=teacher_correct=student_correct=0
    js_sum=0.0
    for sample in records:
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=model(**sample["batch"])
        check_tensor(logits,"student decision logits")
        mask=sample["batch"]["marker_mask"].bool()
        t=torch.softmax(sample["teacher_logits"].masked_fill(~mask,-1e4),-1)
        s=torch.softmax(logits.float().masked_fill(~mask,-1e4),-1)
        ti,si=t.argmax(-1),s.argmax(-1)
        mix=(t+s)*0.5
        divergence=0.5*(t*(t.clamp_min(1e-9).log()-
                           mix.clamp_min(1e-9).log())).sum(-1)
        divergence+=0.5*(s*(s.clamp_min(1e-9).log()-
                            mix.clamp_min(1e-9).log())).sum(-1)
        js_sum+=float(divergence.sum())
        agree+=int((ti==si).sum())
        teacher_correct+=int((ti==sample["gold"]).sum())
        student_correct+=int((si==sample["gold"]).sum())
        n+=len(ti)
    out={"decisions":n,"agreement":agree/max(n,1),
         "teacher_gold_accuracy":teacher_correct/max(n,1),
         "student_gold_accuracy":student_correct/max(n,1),
         "mean_js":js_sum/max(n,1)}
    for k,v in out.items():
        if not math.isfinite(float(v)):
            raise FloatingPointError(f"Nonfinite DEV quality {k}")
    return out
INITIAL_GATE=quality_from_records(student_eval,DEV_GATE_RECORDS)
assert INITIAL_GATE["agreement"]==1.0
print("Native teacher vs same original student on DEV-GATE:",INITIAL_GATE)


In [ ]:
#@title 7. Stage A + B: streaming CeNN pretraining and real-prefix 3→1 teacher distillation
def pretrain_cenn_stream(block,start):
    if CENN_STEPS_PER_GROUP==0:return {"steps":0}
    optimizer=torch.optim.AdamW(block.cenn.parameters(),lr=5e-4)
    generator=fresh_cycle(train_items,SEED+start*17)
    block.cenn.train()
    for step in range(CENN_STEPS_PER_GROUP):
        rec=capture_triplet(start,next(generator),include_rollout=False)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            prediction=block.cellular_only_prediction(rec["x"],rec["valid"])
            loss,_,_=pair_loss(prediction,rec["y_first"],rec["valid"])
        require_finite(loss,"CeNN warmup")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(block.cenn.parameters(),1.)
        optimizer.step()
    del optimizer,generator
    block.eval()
    return {"steps":CENN_STEPS_PER_GROUP}
def distill_stream(block,start,probe):
    block.requires_grad_(True)
    qkv,other=[],[]
    for name,weight in block.named_parameters():
        (qkv if name.startswith(("qkv1.","qkv2.")) else other).append(weight)
    optimizer=torch.optim.AdamW([
        {"params":qkv,"lr":QKV_LR,"start_lr":QKV_LR},
        {"params":other,"lr":GROUP_LR,"start_lr":GROUP_LR},
    ],weight_decay=2e-4)
    generator=fresh_cycle(train_items,SEED+start*31+13)
    b0=local_probe(block,probe,input_key="x")
    r0=local_probe(block,probe,input_key="rollout")
    best_rank=local_rank(b0,r0)
    best_state=cpu_snapshot(block)
    best_step=0
    history=[{"step":0,"original":b0,"rollout":r0}]
    for step in range(1,STEPS_PER_GROUP+1):
        rec=capture_triplet(start,next(generator),include_rollout=True)
        block.train() # populates differentiable first-stage last_mid
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            output=block(rec["rollout"],attention_mask=rec["valid"],
                         position_ids=rec["position_ids"])[0]
            main,_,_=pair_loss(output,rec["y_end"],rec["valid"])
            mid,_,_=pair_loss(block.last_mid,rec["y_first"],rec["valid"])
            cellular=block.cellular_only_prediction(rec["x"],rec["valid"])
            aux,_,_=pair_loss(cellular,rec["y_first"],rec["valid"])
            loss=main+INTERMEDIATE_WEIGHT*mid+CENN_AUX_WEIGHT*aux
        require_finite(loss,"streamed true-prefix triple distillation")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(block.parameters(),1.)
        multiplier=0.1+0.9*0.5*(1+math.cos(math.pi*step/STEPS_PER_GROUP))
        for group in optimizer.param_groups:
            group["lr"]=group["start_lr"]*multiplier
        optimizer.step()
        if step==1 or step%PROBE_EVERY==0 or step==STEPS_PER_GROUP:
            ob=local_probe(block,probe,input_key="x")
            rb=local_probe(block,probe,input_key="rollout")
            rank=local_rank(ob,rb)
            history.append({"step":step,"original":ob,"rollout":rb})
            if rank<best_rank:
                best_rank,best_step=rank,step
                best_state=cpu_snapshot(block)
            print(f"  distill group {start}-{start+2} step {step}: "
                  f"teacher cos {ob['cosine']:.5f}; REAL prefix cos "
                  f"{rb['cosine']:.5f}; best step {best_step}")
    block.load_state_dict(best_state,strict=True)
    block.eval().requires_grad_(False)
    del optimizer,generator,best_state
    return {"steps":STEPS_PER_GROUP,"best_step":best_step,
            "original_probe":local_probe(block,probe,input_key="x"),
            "rollout_probe":local_probe(block,probe,input_key="rollout"),
            "history":history,
            "fresh_teacher_batches":True}


In [ ]:
#@title 8. Task-aware recovery INSIDE currently changed full encoder
# Only the just-installed group is trainable; original teacher, previously
# accepted groups, other native layers and the task head remain frozen.
def selection_rank(q):
    return (3.0*(1-q["agreement"])+
            1.2*(1-q["student_gold_accuracy"])+2.0*q["mean_js"])
def recover_in_changed_model(block,start):
    initial=quality_from_records(student_eval,DEV_GATE_RECORDS)
    if FEEDBACK_STEPS_PER_GROUP==0:
        return {"steps":0,"best_step":0,"before":initial,
                "selected":initial,"history":[]}
    student_eval.eval()
    block.requires_grad_(True)
    optimizer=torch.optim.AdamW(
        [p for p in block.parameters() if p.requires_grad],
        lr=FEEDBACK_LR,weight_decay=5e-5)
    generator=fresh_cycle(train_items,SEED+start*101+71)
    selected=initial
    best_step=0
    best_rank=selection_rank(initial)
    best_state=cpu_snapshot(block)
    history=[{"step":0,"dev_gate":initial}]
    stale=0
    for step in range(1,FEEDBACK_STEPS_PER_GROUP+1):
        chosen=next(generator)
        inp=direct_batch(chosen)
        gold=torch.tensor([z["gold_index"] for z in chosen],
                          device=device,dtype=torch.long)
        # Alignment targets: actual FINAL encoder hidden vectors,
        # not merely the three isolated group targets.
        seen={}
        def teacher_hidden(mod,args,out):
            seen["teacher_h"]=out.last_hidden_state.detach()
        def student_hidden(mod,args,out):
            seen["student_h"]=out.last_hidden_state
        ht=teacher.encoder.register_forward_hook(teacher_hidden)
        hs=student_eval.encoder.register_forward_hook(student_hidden)
        optimizer.zero_grad(set_to_none=True)
        try:
            with torch.no_grad(),torch.autocast(
                    "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
                teacher_logits,teacher_action=teacher(**inp)
            with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
                student_logits,student_action=student_eval(**inp)
                ce=F.cross_entropy(student_logits.float(),gold)
                kd=decision_distill_loss(
                    student_logits,teacher_logits,inp["marker_mask"].bool(),
                    student_action,teacher_action)
                hidden,_,_=pair_loss(
                    seen["student_h"],seen["teacher_h"],
                    inp["attention_mask"].bool())
                loss=(kd+0.25*ce+HIDDEN_ALIGN_WEIGHT*hidden)
            require_finite(loss,"Changed-student supervised+KD+hidden loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(block.parameters(),0.5)
            optimizer.step()
        finally:
            ht.remove()
            hs.remove()
            seen.clear()
        if step==1 or step%FEEDBACK_EVERY==0 or step==FEEDBACK_STEPS_PER_GROUP:
            quality=quality_from_records(student_eval,DEV_GATE_RECORDS)
            history.append({"step":step,"dev_gate":quality})
            rank=selection_rank(quality)
            if (rank < best_rank and
                quality["student_gold_accuracy"]>=
                    initial["student_gold_accuracy"]-MAX_GOLD_DROP):
                best_rank,best_step=rank,step
                selected=quality
                best_state=cpu_snapshot(block)
                stale=0
            else:
                stale+=1
            print(f"  full-student recovery {start}-{start+2} step "
                  f"{step}: agreement {quality['agreement']:.4%}, "
                  f"gold {quality['student_gold_accuracy']:.4%}, "
                  f"best step {best_step}")
            if stale>=3 and step>=FEEDBACK_EVERY*3:
                print("  Early stop: three consecutive DEV-GATE non-improvements")
                break
    block.load_state_dict(best_state,strict=True)
    block.eval().requires_grad_(False)
    selected=quality_from_records(student_eval,DEV_GATE_RECORDS)
    del optimizer,generator,best_state
    gc.collect()
    torch.cuda.empty_cache()
    return {"steps":step,"best_step":best_step,"before":initial,
            "selected":selected,"history":history,
            "head_frozen":True,"prior_groups_frozen":True}
@torch.inference_mode()
def paired_cuda_ms(model,inp,repeats=BENCH_REPEATS):
    model.eval()
    with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
        for _ in range(4):model(**inp)
        torch.cuda.synchronize()
        timings=[]
        for _ in range(repeats):
            a=torch.cuda.Event(enable_timing=True)
            b=torch.cuda.Event(enable_timing=True)
            a.record()
            model(**inp)
            b.record()
            b.synchronize()
            timings.append(float(a.elapsed_time(b)))
    return {"median_ms":float(np.median(timings)),
            "p95_ms":float(np.percentile(timings,95)),
            "repeats":repeats}


In [ ]:
#@title 9. SELECTIVE 3→1 training, evaluate actual changed model, COMMIT or ROLLBACK
# NOTE: candidate decision is on DEV-GATE, never the independent HOLDOUT.
ACCEPTED_GROUPS=[]
TRIALS=[]
check_batch=DEV_GATE_RECORDS[0]["batch"]
baseline=quality_from_records(student_eval,DEV_GATE_RECORDS)
for start,middle,end in GROUPS:
    group=(start,middle,end)
    print("\n"+"="*86)
    print("Candidate triple",group,
          "| already accepted:",[list(g) for g in ACCEPTED_GROUPS])
    print("="*86)
    # Capture stable real-prefix validation examples BEFORE installation.
    probe=[capture_triplet(start,probe_items[i*BATCH:(i+1)*BATCH])
           for i in range(PROBE_BATCHES)]
    block=JevCeNNGroupSuperLayer(
        teacher.encoder.layers[start],teacher.encoder.layers[middle],
        local_window=LOCAL_WINDOW,
        local_radius=LOCAL_RADIUS,stride=DILATION_STRIDE,
        dilated_radius=DILATED_RADIUS,max_len=MAX_LEN,
        correction_rank=CORRECTION_RANK).to(device)
    warmup=pretrain_cenn_stream(block,start)
    distill=distill_stream(block,start,probe)
    before=quality_from_records(student_eval,DEV_GATE_RECORDS)
    speed_before=paired_cuda_ms(student_eval,check_batch)
    # Apply a prospective new 3-to-1 block, replacing two entire native FFNs.
    originals=install_group(student_eval,group,block)
    feedback=recover_in_changed_model(block,start)
    decision=quality_from_records(student_eval,DEV_GATE_RECORDS)
    real=local_probe(block,probe,input_key="rollout")
    original=local_probe(block,probe,input_key="x")
    # Measure actual native-prefix versus changed-model end-to-end GPU latency.
    speed_after=paired_cuda_ms(student_eval,check_batch)
    speedup=speed_before["median_ms"]/max(speed_after["median_ms"],1e-9)
    verdict=judge_replacement(
        before,decision,local_cosine=real["cosine"],local_nmse=real["nmse"],
        min_agreement=TARGET_AGREEMENT,
        max_incremental_agreement_drop=MAX_INCREMENTAL_AGREEMENT_DROP,
        max_gold_drop=MAX_GOLD_DROP,min_local_cosine=TARGET_COSINE,
        max_local_nmse=TARGET_NMSE,speedup=speedup,
        require_speedup=REQUIRE_CANDIDATE_SPEEDUP)
    if verdict["accepted"]:
        ACCEPTED_GROUPS.append(group)
        weights=OUTPUT_DIR/f"ACCEPTED_group_{start:02d}_{end:02d}_W512.pt"
        torch.save({
            "architecture":block.architecture,"indices":list(group),
            "local_window":LOCAL_WINDOW,"stride":DILATION_STRIDE,
            "correction_rank":CORRECTION_RANK,
            "state":cpu_snapshot(block),"verdict":verdict,
            "selected_using":"DEV-GATE (NOT HOLDOUT)",
        },weights)
        print("ACCEPTED",group,"checkpoint:",weights)
        outcome="accepted"
    else:
        restore_group(student_eval,group,originals)
        del block
        weights=None
        restored=quality_from_records(student_eval,DEV_GATE_RECORDS)
        assert abs(restored["agreement"]-before["agreement"])<1e-9,(
            "ROLLBACK FAILED: original encoder teacher agreement changed!")
        assert abs(restored["student_gold_accuracy"]-
                   before["student_gold_accuracy"])<1e-9
        print("REJECTED and ROLLED BACK",group,"reasons:",verdict["reasons"])
        outcome="rejected"
    trial={
        "group":list(group),"status":outcome,
        "warmup":warmup,"distillation":distill,"feedback":feedback,
        "original_input_probe":original,"real_prefix_probe":real,
        "before_cumulative_quality":before,
        "proposed_cumulative_quality":decision,
        "gate_verdict":verdict,
        "native_prefix_cuda":speed_before,
        "proposed_cuda":speed_after,
        "candidate_speedup":speedup,
        "checkpoint":None if weights is None else str(weights),
    }
    TRIALS.append(trial)
    baseline=quality_from_records(student_eval,DEV_GATE_RECORDS)
    assert baseline["agreement"]>=TARGET_AGREEMENT-1e-12,(
        "Selective gate permitted cumulative degradation!")
    progress=OUTPUT_DIR/"jev_cenn_selective_PROGRESS.json"
    progress.write_text(json.dumps({
        "training_step":"candidate_selection_not_holdout",
        "accepted_groups":[list(g) for g in ACCEPTED_GROUPS],
        "trials":TRIALS,
        "current_DEV_gate":baseline,
    },indent=2,allow_nan=False))
    gc.collect()
    torch.cuda.empty_cache()
FINAL_SELECTED_STRUCTURE=plan_active_layers(28,ACCEPTED_GROUPS)
print("\nSELECTED model structure:",FINAL_SELECTED_STRUCTURE)
print("Final DEV-GATE metrics:",baseline)
